In [ ]:
# Packages
import sys, os, glob
import asdf
import numpy as np
import pandas as pd
from scipy import interpolate
from tqdm.notebook import tqdm as tqdm
import matplotlib.patheffects as path_effects
import warnings
warnings.filterwarnings("ignore")
                        
# Plots
from matplotlib.colors import LogNorm
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy.visualization.mpl_normalize import simple_norm

# personal import 
from ExoCAT.mrs_tools import load_cube_dir, load_spec_dir

# Parameter for plot
fs = 15
bands = ["1A","1B","1C","2A","2B","2C","3A","3B","3C","4A", "4B", "4C"] # Set up the bands


path = '/Users/mmalin/MIRI/MRS/Disks/DATA/J1615/'

## 1. Load the cubes

In [ ]:
# Load data cubes : 
science_obs = load_cube_dir(path+'stage3/', idx=1, suffixe='s3d') 

science_cube, science_wave = {}, {}
for band in bands:
    science_cube[band]  = science_obs[band]['cube'] * 1e6
    science_wave[band] = science_obs[band]['wav']
     

In [ ]:
# Plot the median slice for each cube
fig = plt.figure(figsize = (12,14), tight_layout = False)
for ch in range(len(bands)):
    ax = fig.add_subplot(4, 3, 1+ch)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size = "5%", pad = 0.05)
    median_image = np.nanmedian(science_cube[bands[ch]], axis=0)
    im = ax.imshow(median_image, cmap="inferno",origin='lower', norm=LogNorm(vmin=1))
    cbar = fig.colorbar(im, cax = cax, orientation = "vertical")
    cbar.set_label("Flux [$\mu$Jy]", color="black", size=fs)
    ax.set_title(f"Band {bands[ch]} : {np.min(science_wave[bands[ch]]):.1f} – {np.max(science_wave[bands[ch]]):.1f} $\mu$m", 
                 fontsize=fs)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.tight_layout()
plt.show()

## 2. Load the spectra

In [ ]:
spectra_1d = load_spec_dir(path + 'stage3/', idx=1, suffixe='x1d')

fig, ax = plt.subplots(figsize=(8, 5))

for band in bands:
    ax.errorbar(spectra_1d[band]['wav'], spectra_1d[band]['flux'],
                yerr=spectra_1d[band]['err'], fmt='-', capsize=0,
                alpha=0.9, label=band)

ax.set_xlabel(r"Wavelength [$\mu$m]", fontsize=16)
ax.set_ylabel(r"Flux [Jy]", fontsize=16)

ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax.tick_params(which="major", length=8, width=1.4, labelsize=14)
ax.tick_params(which="minor", length=4, width=1.0)
ax.set_ylim(-0.1, 2)
ax.minorticks_on()
ax.legend(fontsize=12, frameon=False, ncol=4)

fig.tight_layout()
plt.show()